# 04 — Player Projections

This notebook builds 2026 player and position-group projections that feed into the team strength model.

Recent multi-year performance is used to estimate quarterback, skill position, offensive line, and defensive strength. Quarterback is treated separately because of its larger impact on team performance.

All historical features are constructed using only information available before the projected season.

In [1]:
from pathlib import Path

import pandas as pd
import polars as pl
import nflreadpy as nfl

In [2]:
PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

## Load Player Data

The projection process starts with historical player production, playing time, roster information, and team context.

I keep the initial inputs limited to the datasets needed to build the player season modeling table, then add other sources only when they provide a specific projection feature.

In [3]:
player_stats = pl.read_parquet(
    PROCESSED_DIR / "player_stats_clean.parquet"
)

snap_counts = pl.read_parquet(
    PROCESSED_DIR / "snap_counts_clean.parquet"
)

rosters = pl.read_parquet(
    PROCESSED_DIR / "rosters_clean.parquet"
)

draft_picks_clean = pl.read_parquet(
    PROCESSED_DIR / "draft_picks_clean.parquet"
)

In [4]:
roster_bridge = (
    rosters
    .filter(
        pl.col("gsis_id").is_not_null()
    )
    .sort([
        "season",
        "gsis_id",
        "team"
    ])
    .group_by([
        "season",
        "gsis_id"
    ])
    .agg([
        pl.col("pfr_id")
        .drop_nulls()
        .first()
        .alias("pfr_id"),

        pl.col("full_name")
        .drop_nulls()
        .first()
        .alias("roster_name"),

        pl.col("position")
        .drop_nulls()
        .first()
        .alias("roster_position"),

        pl.col("years_exp")
        .drop_nulls()
        .max()
        .alias("years_exp"),

        pl.col("birth_date")
        .drop_nulls()
        .first()
        .alias("birth_date"),

        pl.col("height")
        .drop_nulls()
        .first()
        .alias("height"),

        pl.col("weight")
        .drop_nulls()
        .first()
        .alias("weight"),

        pl.col("college")
        .drop_nulls()
        .first()
        .alias("college"),

        pl.col("team")
        .drop_nulls()
        .unique()
        .sort()
        .alias("roster_teams"),

        pl.col("team")
        .drop_nulls()
        .n_unique()
        .alias("roster_team_count")
    ])
)

In [5]:
player_season_snaps = (
    snap_counts
    .filter(
        pl.col("pfr_player_id").is_not_null()
    )
    .group_by([
        "season",
        "pfr_player_id"
    ])
    .agg([
        pl.col("offense_snaps")
        .fill_null(0)
        .sum()
        .alias("offense_snaps"),

        pl.col("defense_snaps")
        .fill_null(0)
        .sum()
        .alias("defense_snaps"),

        pl.col("st_snaps")
        .fill_null(0)
        .sum()
        .alias("special_teams_snaps"),

        pl.col("game_id")
        .n_unique()
        .alias("games_with_snaps"),

        pl.col("team")
        .drop_nulls()
        .unique()
        .sort()
        .alias("snap_teams")
    ])
    .with_columns(
        (
            pl.col("offense_snaps")
            + pl.col("defense_snaps")
            + pl.col("special_teams_snaps")
        ).alias("total_snaps")
    )
)

In [6]:
player_seasons = (
    player_stats
    .join(
        roster_bridge,
        left_on=["season", "player_id"],
        right_on=["season", "gsis_id"],
        how="left"
    )
    .join(
        player_season_snaps,
        left_on=["season", "pfr_id"],
        right_on=["season", "pfr_player_id"],
        how="left"
    )
)

## Standardize Projection Positions

Player projections differ substantially by position because production, playing time, aging, and team impact are not comparable across roles.

I standardize players into projection groups before creating historical performance features. Quarterbacks are kept separate because of their outsized impact on team performance, while offensive and defensive players are grouped by their primary role.

In [7]:
player_seasons = (
    player_seasons
    .with_columns(
        pl.when(pl.col("position") == "QB")
        .then(pl.lit("QB"))

        .when(pl.col("position").is_in(["RB", "FB"]))
        .then(pl.lit("RB"))

        .when(pl.col("position") == "WR")
        .then(pl.lit("WR"))

        .when(pl.col("position") == "TE")
        .then(pl.lit("TE"))

        .when(pl.col("position").is_in([
            "T", "OT", "G", "OG", "C", "OL"
        ]))
        .then(pl.lit("OL"))

        .when(pl.col("position").is_in([
            "DE", "DT", "NT", "DL"
        ]))
        .then(pl.lit("DL"))

        .when(pl.col("position").is_in([
            "LB", "ILB", "OLB", "MLB"
        ]))
        .then(pl.lit("LB"))

        .when(pl.col("position").is_in([
            "CB", "DB"
        ]))
        .then(pl.lit("CB"))

        .when(pl.col("position").is_in([
            "S", "FS", "SS", "SAF"
        ]))
        .then(pl.lit("S"))

        .when(pl.col("position").is_in([
            "K", "P"
        ]))
        .then(pl.lit("ST"))

        .otherwise(pl.lit("OTHER"))
        .alias("projection_position")
    )
)

## Age and Experience

Age and NFL experience provide important context for projecting future player performance.

Players at different stages of their careers can have different development, stability, and decline patterns. These variables are calculated at the player season level so they can later be shifted appropriately when constructing leakage safe future season projections.

In [8]:
player_seasons = (
    player_seasons
    .with_columns(
        pl.date(
            pl.col("season"),
            pl.lit(9),
            pl.lit(1)
        ).alias("season_start_date")
    )
    .with_columns(
        (
            (
                pl.col("season_start_date")
                - pl.col("birth_date")
            ).dt.total_days()
            / 365.25
        ).alias("age")
    )
)

## Leakage Safe Projection Framework

Historical player statistics describe what occurred during each completed NFL season. To use these data for forecasting, each player season is converted into a future target season observation.

For a target season Y, predictor variables are restricted to information available before season Y. Historical performance is represented through lagged features from Y-1, Y-2, and Y-3, while performance during season Y is reserved as the prediction target.

This structure prevents future season information from leaking into historical model training and allows the same framework to be used for walk-forward validation and eventual 2026 projections.

In [9]:
player_seasons = (
    player_seasons
    .sort(["player_id", "season"])
)

## Historical Player Lags

Player projections use exact prior season information rather than simply taking the previous observed row.

For a target season Y:

- Lag 1 represents Y-1
- Lag 2 represents Y-2
- Lag 3 represents Y-3

If a player did not record a season in one of those years, the corresponding lag remains missing. This prevents older performance from being incorrectly treated as the immediately previous season.

In [10]:
player_history_source = (
    player_seasons
    .select([
        "season",
        "player_id",
        "player_display_name",
        "projection_position",
        "recent_team",
        "games",
        "offense_snaps",
        "defense_snaps",
        "total_snaps",

        "attempts",
        "passing_epa",
        "passing_cpoe",
        "sacks_suffered",

        "carries",
        "rushing_yards",
        "rushing_epa",

        "targets",
        "receiving_yards",
        "receiving_epa",
        "target_share",

        "def_tackles_solo",
        "def_sacks",
        "def_qb_hits",
        "def_interceptions",
        "def_pass_defended"
    ])
)

In [11]:
projection_rows = (
    player_seasons
    .select([
        "player_id",
        "player_display_name",
        "projection_position",
        "season"
    ])
    .rename({
        "season": "target_season"
    })
)

In [12]:
def make_lag_table(df, lag):
    rename_map = {
        col: f"lag{lag}_{col}"
        for col in df.columns
        if col not in ["season", "player_id"]
    }

    return (
        df
        .with_columns(
            (pl.col("season") + lag)
            .alias("target_season")
        )
        .drop("season")
        .rename(rename_map)
    )

In [13]:
lag1 = make_lag_table(player_history_source, 1)
lag2 = make_lag_table(player_history_source, 2)
lag3 = make_lag_table(player_history_source, 3)

projection_rows = (
    projection_rows
    .join(
        lag1,
        on=["player_id", "target_season"],
        how="left"
    )
    .join(
        lag2,
        on=["player_id", "target_season"],
        how="left"
    )
    .join(
        lag3,
        on=["player_id", "target_season"],
        how="left"
    )
)

## Recent Performance

Recent performance is generally more informative for projection than older performance, but relying on only one season can make projections too sensitive to short term variance.

I combine up to three prior seasons with greater weight on the most recent year. Missing seasons are not treated as zero production; the available weights are rescaled while separate history availability features preserve information about gaps in a player's career.

In [14]:
RECENCY_WEIGHTS = {
    1: 0.60,
    2: 0.25,
    3: 0.15
}


def weighted_history_expr(feature):
    numerator = (
        pl.col(f"lag1_{feature}").fill_null(0) * RECENCY_WEIGHTS[1]
        + pl.col(f"lag2_{feature}").fill_null(0) * RECENCY_WEIGHTS[2]
        + pl.col(f"lag3_{feature}").fill_null(0) * RECENCY_WEIGHTS[3]
    )

    denominator = (
        pl.col(f"lag1_{feature}").is_not_null().cast(pl.Float64)
        * RECENCY_WEIGHTS[1]
        + pl.col(f"lag2_{feature}").is_not_null().cast(pl.Float64)
        * RECENCY_WEIGHTS[2]
        + pl.col(f"lag3_{feature}").is_not_null().cast(pl.Float64)
        * RECENCY_WEIGHTS[3]
    )

    return (
        pl.when(denominator > 0)
        .then(numerator / denominator)
        .otherwise(None)
        .alias(f"weighted_{feature}")
    )

## Quarterback Projections

Quarterback is treated separately because passing efficiency has a large effect on team performance.

The projection uses the previous three seasons of passing EPA per dropback, weighting recent seasons more heavily. Small historical samples are regressed toward the league average so short stretches of unusually strong or weak play do not dominate the forecast.

The latest 2026 depth chart is then used to identify each team's expected starting quarterback.

In [15]:
qb_projection_rows = (
    projection_rows
    .filter(
        pl.col("projection_position") == "QB"
    )
)

### Historical Passing Efficiency

Passing efficiency is measured as EPA per dropback rather than total passing EPA so quarterbacks with different workloads can be compared on the same scale.

Each historical season receives a 60/25/15 recency weight, with the most recent season receiving the greatest influence.

In [16]:
for lag in [1, 2, 3]:
    qb_projection_rows = qb_projection_rows.with_columns([
        (
            pl.col(f"lag{lag}_attempts").fill_null(0)
            + pl.col(f"lag{lag}_sacks_suffered").fill_null(0)
        ).alias(f"lag{lag}_dropbacks"),

        pl.when(
            (
                pl.col(f"lag{lag}_attempts").fill_null(0)
                + pl.col(f"lag{lag}_sacks_suffered").fill_null(0)
            ) > 0
        )
        .then(
            pl.col(f"lag{lag}_passing_epa").fill_null(0)
            / (
                pl.col(f"lag{lag}_attempts").fill_null(0)
                + pl.col(f"lag{lag}_sacks_suffered").fill_null(0)
            )
        )
        .otherwise(None)
        .alias(f"lag{lag}_epa_per_dropback")
    ])

qb_projection_rows = qb_projection_rows.with_columns([
    (
        RECENCY_WEIGHTS[1] * pl.col("lag1_passing_epa").fill_null(0)
        + RECENCY_WEIGHTS[2] * pl.col("lag2_passing_epa").fill_null(0)
        + RECENCY_WEIGHTS[3] * pl.col("lag3_passing_epa").fill_null(0)
    ).alias("weighted_passing_epa"),

    (
        RECENCY_WEIGHTS[1] * pl.col("lag1_dropbacks").fill_null(0)
        + RECENCY_WEIGHTS[2] * pl.col("lag2_dropbacks").fill_null(0)
        + RECENCY_WEIGHTS[3] * pl.col("lag3_dropbacks").fill_null(0)
    ).alias("weighted_dropbacks")
])

qb_projection_rows = qb_projection_rows.with_columns([
    pl.when(pl.col("weighted_dropbacks") > 0)
    .then(
        pl.col("weighted_passing_epa")
        / pl.col("weighted_dropbacks")
    )
    .otherwise(None)
    .alias("weighted_qb_epa_per_dropback")
])

### Quarterback Efficiency Regression

Quarterback efficiency can be highly volatile over small samples. A player with only a handful of historical pass attempts should not receive the same confidence as an established starter with several seasons of data.

To account for this, historical quarterback efficiency is regressed toward league average performance based on the amount of prior passing volume available.

League baselines are calculated using only seasons before each target season to prevent future information from leaking into the projection features.

In [17]:
qb_league_baselines = (
    player_history_source
    .filter(
        pl.col("projection_position") == "QB"
    )
    .with_columns([
        (
            pl.col("attempts").fill_null(0)
            + pl.col("sacks_suffered").fill_null(0)
        ).alias("dropbacks")
    ])
    .group_by("season")
    .agg([
        pl.col("passing_epa").sum().alias("league_passing_epa"),
        pl.col("dropbacks").sum().alias("league_dropbacks")
    ])
    .with_columns([
        (
            pl.col("league_passing_epa")
            / pl.col("league_dropbacks")
        ).alias("league_qb_epa_per_dropback")
    ])
    .select([
        (pl.col("season") + 1).alias("target_season"),
        "league_qb_epa_per_dropback"
    ])
)

### Regression Toward League Average

Historical QB efficiency is more reliable when it comes from a larger number of dropbacks.

The weighted historical EPA per dropback is therefore blended with the league average using prior workload as a simple reliability measure. Quarterbacks with extensive history remain close to their observed performance, while small-sample players are pulled more strongly toward league average.

In [18]:
QB_REGRESSION_DROPBACKS = 300

qb_projection_rows = (
    qb_projection_rows
    .join(
        qb_league_baselines,
        on="target_season",
        how="left"
    )
    .with_columns([
        (
            pl.col("weighted_dropbacks")
            / (
                pl.col("weighted_dropbacks")
                + QB_REGRESSION_DROPBACKS
            )
        ).alias("qb_reliability")
    ])
)

In [19]:
qb_projection_rows = qb_projection_rows.with_columns([
    (
        pl.col("qb_reliability")
        * pl.col("weighted_qb_epa_per_dropback")
            .fill_null(pl.col("league_qb_epa_per_dropback"))
        + (
            1 - pl.col("qb_reliability")
        )
        * pl.col("league_qb_epa_per_dropback")
    ).alias("projected_qb_epa_per_dropback")
])

### 2026 Quarterback Projection

The 2026 quarterback projection uses the same simple framework as the historical backtest.

Recent passing efficiency is calculated from the previous three seasons using 60/25/15 weights, then regressed toward the league average based on historical dropback volume. Current roster and depth chart data determine each player's 2026 team and identify the expected starter.

In [20]:
current_2026_rosters = nfl.load_rosters([2026])
current_2026_draft = nfl.load_draft_picks([2026])

In [21]:
qb_2026_roster = (
    current_2026_rosters
    .filter(
        (pl.col("position") == "QB")
        & pl.col("gsis_id").is_not_null()
    )
    .sort([
        "gsis_id",
        "week"
    ])
    .group_by("gsis_id")
    .agg([
        pl.col("team").last().alias("team"),
        pl.col("full_name").last().alias("player_display_name"),
        pl.col("birth_date").last().alias("birth_date"),
        pl.col("years_exp").last().alias("target_years_exp"),
        pl.col("status").last().alias("roster_status"),
        pl.col("week").last().alias("roster_week"),
        pl.col("rookie_year").last().alias("rookie_year"),
        pl.col("draft_number").last().alias("roster_draft_pick")
    ])
    .rename({
        "gsis_id": "player_id"
    })
    .with_columns(
        pl.lit(2026).alias("target_season")
    )
)

In [22]:
qb_2026_history_features = (
    player_history_source
    .filter(
        pl.col("projection_position") == "QB"
    )
    .select([
        "player_id",
        "season",
        "attempts",
        "passing_epa",
        "sacks_suffered"
    ])
)

In [23]:
qb_2026_rows = qb_2026_roster

for lag, source_season in [
    (1, 2025),
    (2, 2024),
    (3, 2023)
]:
    lag_data = (
        qb_2026_history_features
        .filter(
            pl.col("season") == source_season
        )
        .select([
            "player_id",
            "attempts",
            "passing_epa",
            "sacks_suffered"
        ])
        .rename({
            "attempts": f"lag{lag}_attempts",
            "passing_epa": f"lag{lag}_passing_epa",
            "sacks_suffered": f"lag{lag}_sacks_suffered"
        })
    )

    qb_2026_rows = qb_2026_rows.join(
        lag_data,
        on="player_id",
        how="left"
    )

In [24]:
qb_2026_draft = (
    current_2026_draft
    .filter(
        (pl.col("position") == "QB")
        & pl.col("gsis_id").is_not_null()
    )
    .select([
        pl.col("gsis_id").alias("player_id"),
        pl.col("season").alias("draft_season"),
        pl.col("round").alias("draft_round"),
        pl.col("pick").alias("draft_pick")
    ])
    .unique(
        subset=["player_id"]
    )
)

qb_2026_rows = (
    qb_2026_rows
    .join(
        qb_2026_draft,
        on="player_id",
        how="left"
    )
)

In [25]:
qb_2026_rows = qb_2026_rows.with_columns([
    pl.coalesce([
        pl.col("draft_pick"),
        pl.col("roster_draft_pick")
    ]).alias("draft_pick"),

    pl.when(
        pl.col("draft_season").is_not_null()
    )
    .then(
        pl.col("draft_season")
    )
    .otherwise(
        pl.col("rookie_year")
    )
    .alias("draft_season")
])

In [26]:
qb_2026_rows = qb_2026_rows.with_columns([
    pl.col("birth_date")
    .cast(pl.Date, strict=False)
    .alias("birth_date")
])

qb_2026_rows = qb_2026_rows.with_columns([
    (
        (
            pl.date(2026, 9, 1)
            - pl.col("birth_date")
        ).dt.total_days()
        / 365.2425
    ).alias("target_age")
])

In [27]:
qb_2026_rows = qb_2026_rows.with_columns([
    (
        pl.col("target_years_exp") == 0
    ).cast(pl.Int8).alias("rookie_qb")
])

In [28]:
for lag in [1, 2, 3]:
    qb_2026_rows = qb_2026_rows.with_columns([
        (
            pl.col(f"lag{lag}_attempts").fill_null(0)
            + pl.col(f"lag{lag}_sacks_suffered").fill_null(0)
        ).alias(f"lag{lag}_dropbacks")
    ])

qb_2026_rows = qb_2026_rows.with_columns([
    (
        RECENCY_WEIGHTS[1] * pl.col("lag1_passing_epa").fill_null(0)
        + RECENCY_WEIGHTS[2] * pl.col("lag2_passing_epa").fill_null(0)
        + RECENCY_WEIGHTS[3] * pl.col("lag3_passing_epa").fill_null(0)
    ).alias("weighted_passing_epa"),

    (
        RECENCY_WEIGHTS[1] * pl.col("lag1_dropbacks")
        + RECENCY_WEIGHTS[2] * pl.col("lag2_dropbacks")
        + RECENCY_WEIGHTS[3] * pl.col("lag3_dropbacks")
    ).alias("weighted_dropbacks")
])

league_2026_qb_epa = (
    player_history_source
    .filter(
        (pl.col("season") == 2025)
        & (pl.col("projection_position") == "QB")
    )
    .with_columns([
        (
            pl.col("attempts").fill_null(0)
            + pl.col("sacks_suffered").fill_null(0)
        ).alias("dropbacks")
    ])
    .select(
        pl.col("passing_epa").sum()
        / pl.col("dropbacks").sum()
    )
    .item()
)

qb_2026_rows = qb_2026_rows.with_columns([
    pl.when(pl.col("weighted_dropbacks") > 0)
    .then(
        pl.col("weighted_passing_epa")
        / pl.col("weighted_dropbacks")
    )
    .otherwise(league_2026_qb_epa)
    .alias("weighted_qb_epa_per_dropback"),

    (
        pl.col("weighted_dropbacks")
        / (
            pl.col("weighted_dropbacks")
            + QB_REGRESSION_DROPBACKS
        )
    ).alias("qb_reliability")
])

qb_2026_rows = qb_2026_rows.with_columns([
    (
        pl.col("qb_reliability")
        * pl.col("weighted_qb_epa_per_dropback")
        + (
            1 - pl.col("qb_reliability")
        ) * league_2026_qb_epa
    ).alias("projected_qb_epa_per_dropback")
])

In [29]:
qb_2026_projections = (
    qb_2026_rows
    .select([
        "player_id",
        "player_display_name",
        "team",
        "roster_status",
        "roster_week",
        "target_age",
        "target_years_exp",
        "rookie_qb",
        "draft_pick",
        "weighted_dropbacks",
        "projected_qb_epa_per_dropback"
    ])
    .to_pandas()
)

### Current QB Role and Depth Chart

Historical performance estimates quarterback quality, but current roster decisions determine who is expected to start.

The latest available 2026 depth chart is therefore used to identify each team's QB1 and to resolve current team assignments.

In [30]:
current_2026_depth_charts = nfl.load_depth_charts([2026])

In [31]:
latest_depth_chart_dt = (
    current_2026_depth_charts
    .select(pl.col("dt").max())
    .item()
)

qb_2026_depth = (
    current_2026_depth_charts
    .filter(
        (pl.col("dt") == latest_depth_chart_dt)
        & (pl.col("pos_abb") == "QB")
        & pl.col("gsis_id").is_not_null()
    )
    .select([
        pl.col("gsis_id").alias("player_id"),
        pl.col("team").alias("depth_chart_team"),
        pl.col("pos_rank").alias("qb_depth_rank")
    ])
    .unique(subset=["player_id"])
)

In [32]:
qb_2026_projections = (
    qb_2026_projections
    .merge(
        qb_2026_depth.to_pandas(),
        on="player_id",
        how="left"
    )
)

qb_2026_projections["qb_depth_rank"] = (
    qb_2026_projections["qb_depth_rank"]
    .fillna(99)
    .astype(int)
)

qb_2026_projections["current_qb1"] = (
    qb_2026_projections["qb_depth_rank"] == 1
).astype(int)

In [33]:
qb_2026_projections["team"] = (
    qb_2026_projections["depth_chart_team"]
    .fillna(qb_2026_projections["team"])
)

In [34]:
qb_2026_team_qb1 = (
    qb_2026_projections[
        qb_2026_projections["current_qb1"] == 1
    ]
    .copy()
    .sort_values("team")
    .reset_index(drop=True)
)

### Depth Chart Based Starter Assignment

The statistical projection estimates quarterback performance, while the current depth chart determines each team's expected starter.

This prevents historical playing time from overriding current roster information such as trades, free agency, or preseason competitions.

### Final 2026 QB Output

The final quarterback table contains one projected starter per team.

Each starter receives a projected EPA per dropback based on recent historical performance, regression toward league average, and the latest available 2026 depth chart information. This team level quarterback rating will feed directly into the team strength model.

In [35]:
qb_2026_team_strength = (
    qb_2026_team_qb1[
        [
            "team",
            "player_id",
            "player_display_name",
            "target_age",
            "rookie_qb",
            "projected_qb_epa_per_dropback"
        ]
    ]
    .rename(columns={
        "player_id": "qb1_player_id",
        "player_display_name": "qb1_name",
        "target_age": "qb1_age",
        "rookie_qb": "qb1_rookie",
        "projected_qb_epa_per_dropback":
            "qb1_projected_epa_per_dropback"
    })
    .sort_values(
        "qb1_projected_epa_per_dropback",
        ascending=False
    )
    .reset_index(drop=True)
)

print("QB1 rows:", len(qb_2026_team_strength))
print("Teams:", qb_2026_team_strength["team"].nunique())

qb_2026_team_strength

QB1 rows: 32
Teams: 32


,team,qb1_player_id,qb1_name,qb1_age,qb1_rookie,qb1_projected_epa_per_dropback
0,NE,00-0039851,Drake Maye,24.005969,0,0.141987
1,DET,00-0033106,Jared Goff,31.882927,0,0.140400
2,GB,00-0036264,Jordan Love,27.830825,0,0.134269
3,SF,00-0037834,Brock Purdy,26.680904,0,0.131664
4,LA,00-0026498,Matthew Stafford,38.566158,0,0.130037
5,BUF,00-0034857,Josh Allen,30.281251,0,0.120673
6,DAL,00-0033077,Dak Prescott,33.093082,0,0.109471
7,BAL,00-0034796,Lamar Jackson,29.648795,0,0.102775
8,MIA,00-0038128,Malik Willis,27.272292,0,0.088035
9,SEA,00-0034869,Sam Darnold,29.240847,0,0.087481


In [36]:
qb_2026_team_strength.to_parquet(
    PROCESSED_DIR / "2026_qb_strength.parquet",
    index=False
)

## Offensive Skill Position Projections

Running backs, wide receivers, and tight ends are projected using a simpler approach than quarterbacks.

For these positions, the goal is not to predict an exact box score for every player. Instead, recent rushing and receiving production are used to estimate the amount of offensive talent each player brings into the 2026 season.

The player projections will later be aggregated into team level RB, WR, and TE ratings for the team strength model.

In [37]:
skill_projection_rows = (
    projection_rows
    .filter(
        pl.col("projection_position").is_in([
            "RB",
            "WR",
            "TE"
        ])
    )
)

In [38]:
skill_projection_rows = skill_projection_rows.with_columns([
    weighted_history_expr("rushing_yards")
    .alias("weighted_rushing_yards"),

    weighted_history_expr("rushing_epa")
    .alias("weighted_rushing_epa"),

    weighted_history_expr("receiving_yards")
    .alias("weighted_receiving_yards"),

    weighted_history_expr("receiving_epa")
    .alias("weighted_receiving_epa"),

    weighted_history_expr("targets")
    .alias("weighted_targets")
])

In [39]:
print(
    skill_projection_rows
    .filter(
        (pl.col("target_season") == 2025)
        & pl.col("weighted_receiving_yards").is_not_null()
    )
    .select([
        "player_display_name",
        "projection_position",
        "weighted_rushing_yards",
        "weighted_receiving_yards",
        "weighted_receiving_epa",
        "weighted_targets"
    ])
    .sort(
        "weighted_receiving_yards",
        descending=True,
        nulls_last=True
    )
    .head(20)
)

shape: (20, 6)
┌────────────────┬────────────────┬────────────────┬───────────────┬───────────────┬───────────────┐
│ player_display ┆ projection_pos ┆ weighted_rushi ┆ weighted_rece ┆ weighted_rece ┆ weighted_targ │
│ _name          ┆ ition          ┆ ng_yards       ┆ iving_yards   ┆ iving_epa     ┆ ets           │
│ ---            ┆ ---            ┆ ---            ┆ ---           ┆ ---           ┆ ---           │
│ str            ┆ str            ┆ f64            ┆ f64           ┆ f64           ┆ f64           │
╞════════════════╪════════════════╪════════════════╪═══════════════╪═══════════════╪═══════════════╡
│ Ja'Marr Chase  ┆ WR             ┆ 18.9           ┆ 1485.7        ┆ 65.888596     ┆ 161.35        │
│ Justin         ┆ WR             ┆ 2.4            ┆ 1459.65       ┆ 60.527049     ┆ 145.0         │
│ Jefferson      ┆                ┆                ┆               ┆               ┆               │
│ CeeDee Lamb    ┆ WR             ┆ 77.3           ┆ 1357.5        ┆ 47.0199

### Skill Position Production Scores

Running backs, wide receivers, and tight ends are evaluated using recent rushing and receiving production.

Because the positions contribute differently, each group uses a slightly different mix of rushing and receiving value. The goal is to create a simple player score that can later be aggregated into team position-group strength.

In [40]:
skill_projection_rows = skill_projection_rows.with_columns([
    pl.when(
        pl.col("projection_position") == "RB"
    )
    .then(
        0.45 * pl.col("weighted_rushing_yards").fill_null(0)
        + 4.0 * pl.col("weighted_rushing_epa").fill_null(0)
        + 0.20 * pl.col("weighted_receiving_yards").fill_null(0)
        + 2.0 * pl.col("weighted_receiving_epa").fill_null(0)
    )
    .when(
        pl.col("projection_position") == "WR"
    )
    .then(
        0.10 * pl.col("weighted_rushing_yards").fill_null(0)
        + 0.55 * pl.col("weighted_receiving_yards").fill_null(0)
        + 4.0 * pl.col("weighted_receiving_epa").fill_null(0)
    )
    .when(
        pl.col("projection_position") == "TE"
    )
    .then(
        0.50 * pl.col("weighted_receiving_yards").fill_null(0)
        + 4.0 * pl.col("weighted_receiving_epa").fill_null(0)
    )
    .otherwise(0)
    .alias("skill_production_score")
])

In [41]:
position_score_stats = (
    skill_projection_rows
    .filter(
        pl.col("skill_production_score").is_not_null()
    )
    .group_by("projection_position")
    .agg([
        pl.col("skill_production_score")
        .mean()
        .alias("position_score_mean"),

        pl.col("skill_production_score")
        .std()
        .alias("position_score_std")
    ])
)

skill_projection_rows = (
    skill_projection_rows
    .join(
        position_score_stats,
        on="projection_position",
        how="left"
    )
    .with_columns([
        (
            (
                pl.col("skill_production_score")
                - pl.col("position_score_mean")
            )
            / pl.col("position_score_std")
        ).alias("skill_production_z")
    ])
)

### 2026 Skill Position Players

The historical production scores are now connected to the current 2026 roster.

This allows each player's recent production to follow them to their current team rather than assuming they remained with the team where that production occurred.

In [42]:
skill_2026_roster = (
    current_2026_rosters
    .filter(
        pl.col("position").is_in([
            "RB",
            "WR",
            "TE"
        ])
    )
    .select([
        "gsis_id",
        "full_name",
        "team",
        "position",
        "status"
    ])
    .rename({
        "gsis_id": "player_id",
        "full_name": "player_display_name",
        "position": "projection_position",
        "status": "roster_status"
    })
    .filter(
        pl.col("player_id").is_not_null()
    )
    .unique(
        subset=["player_id"],
        keep="first"
    )
)

In [43]:
skill_2026_projections = skill_2026_roster

for lag, source_season in [
    (1, 2025),
    (2, 2024),
    (3, 2023)
]:
    lag_data = (
        player_history_source
        .filter(
            pl.col("season") == source_season
        )
        .select([
            "player_id",
            "rushing_yards",
            "rushing_epa",
            "receiving_yards",
            "receiving_epa",
            "targets"
        ])
        .rename({
            "rushing_yards": f"lag{lag}_rushing_yards",
            "rushing_epa": f"lag{lag}_rushing_epa",
            "receiving_yards": f"lag{lag}_receiving_yards",
            "receiving_epa": f"lag{lag}_receiving_epa",
            "targets": f"lag{lag}_targets"
        })
    )

    skill_2026_projections = skill_2026_projections.join(
        lag_data,
        on="player_id",
        how="left"
    )

In [44]:
skill_2026_projections = skill_2026_projections.with_columns([
    weighted_history_expr("rushing_yards")
    .alias("weighted_rushing_yards"),

    weighted_history_expr("rushing_epa")
    .alias("weighted_rushing_epa"),

    weighted_history_expr("receiving_yards")
    .alias("weighted_receiving_yards"),

    weighted_history_expr("receiving_epa")
    .alias("weighted_receiving_epa"),

    weighted_history_expr("targets")
    .alias("weighted_targets")
])

In [45]:
skill_2026_projections = skill_2026_projections.with_columns([
    pl.when(
        pl.col("projection_position") == "RB"
    )
    .then(
        0.45 * pl.col("weighted_rushing_yards").fill_null(0)
        + 4.0 * pl.col("weighted_rushing_epa").fill_null(0)
        + 0.20 * pl.col("weighted_receiving_yards").fill_null(0)
        + 2.0 * pl.col("weighted_receiving_epa").fill_null(0)
    )
    .when(
        pl.col("projection_position") == "WR"
    )
    .then(
        0.10 * pl.col("weighted_rushing_yards").fill_null(0)
        + 0.55 * pl.col("weighted_receiving_yards").fill_null(0)
        + 4.0 * pl.col("weighted_receiving_epa").fill_null(0)
    )
    .when(
        pl.col("projection_position") == "TE"
    )
    .then(
        0.50 * pl.col("weighted_receiving_yards").fill_null(0)
        + 4.0 * pl.col("weighted_receiving_epa").fill_null(0)
    )
    .otherwise(0)
    .alias("skill_production_score")
])

In [46]:
print(
    skill_2026_projections
    .filter(
        pl.col("weighted_receiving_yards").is_not_null()
        | pl.col("weighted_rushing_yards").is_not_null()
    )
    .select([
        "player_display_name",
        "team",
        "projection_position",
        "weighted_rushing_yards",
        "weighted_receiving_yards",
        "skill_production_score"
    ])
    .sort(
        "skill_production_score",
        descending=True,
        nulls_last=True
    )
    .head(25)
)

shape: (25, 6)
┌──────────────────┬──────┬──────────────────┬─────────────────┬─────────────────┬─────────────────┐
│ player_display_n ┆ team ┆ projection_posit ┆ weighted_rushin ┆ weighted_receiv ┆ skill_productio │
│ ame              ┆ ---  ┆ ion              ┆ g_yards         ┆ ing_yards       ┆ n_score         │
│ ---              ┆ str  ┆ ---              ┆ ---             ┆ ---             ┆ ---             │
│ str              ┆      ┆ str              ┆ f64             ┆ f64             ┆ f64             │
╞══════════════════╪══════╪══════════════════╪═════════════════╪═════════════════╪═════════════════╡
│ Puka Nacua       ┆ LA   ┆ WR               ┆ 87.85           ┆ 1499.4          ┆ 1204.165915     │
│ Jaxon            ┆ SEA  ┆ WR               ┆ 28.1            ┆ 1452.5          ┆ 1075.04223      │
│ Smith-Njigba     ┆      ┆                  ┆                 ┆                 ┆                 │
│ Amon-Ra St.      ┆ DET  ┆ WR               ┆ 10.5            ┆ 1383.6     

### Team Skill Position Strength

Individual player scores are aggregated into team position group ratings.

To avoid rewarding teams simply for carrying more players, only the primary contributors at each position are included: the top two running backs, top three wide receivers, and top two tight ends.

In [47]:
skill_2026_ranked = (
    skill_2026_projections
    .with_columns([
        pl.col("skill_production_score")
        .rank(
            method="ordinal",
            descending=True
        )
        .over([
            "team",
            "projection_position"
        ])
        .alias("position_rank")
    ])
)

In [48]:
skill_2026_rotation = (
    skill_2026_ranked
    .filter(
        (
            (pl.col("projection_position") == "RB")
            & (pl.col("position_rank") <= 2)
        )
        |
        (
            (pl.col("projection_position") == "WR")
            & (pl.col("position_rank") <= 3)
        )
        |
        (
            (pl.col("projection_position") == "TE")
            & (pl.col("position_rank") <= 2)
        )
    )
)

In [49]:
team_skill_strength = (
    skill_2026_rotation
    .group_by([
        "team",
        "projection_position"
    ])
    .agg([
        pl.col("skill_production_score")
        .sum()
        .alias("position_group_score")
    ])
    .pivot(
        values="position_group_score",
        index="team",
        on="projection_position"
    )
    .rename({
        "RB": "rb_strength",
        "WR": "wr_strength",
        "TE": "te_strength"
    })
    .sort("team")
)

### Standardized Team Skill Ratings

The raw RB, WR, and TE scores operate on different scales, so each position group rating is standardized across the league.

A value above zero represents an above average projected position group, while a value below zero represents a below average group.

In [50]:
team_skill_strength = team_skill_strength.with_columns([
    (
        (
            pl.col("rb_strength")
            - pl.col("rb_strength").mean()
        )
        / pl.col("rb_strength").std()
    ).alias("rb_strength_z"),

    (
        (
            pl.col("wr_strength")
            - pl.col("wr_strength").mean()
        )
        / pl.col("wr_strength").std()
    ).alias("wr_strength_z"),

    (
        (
            pl.col("te_strength")
            - pl.col("te_strength").mean()
        )
        / pl.col("te_strength").std()
    ).alias("te_strength_z")
])

In [51]:
team_skill_strength = (
    team_skill_strength
    .select([
        "team",
        "rb_strength_z",
        "wr_strength_z",
        "te_strength_z"
    ])
    .sort("team")
)

In [52]:
team_skill_strength.write_parquet(
    PROCESSED_DIR / "2026_skill_position_strength.parquet"
)

## Offensive Line Projection

Offensive line strength is estimated at the unit level rather than through individual box score statistics.

The 2026 rating uses each team's most recent pass protection performance. Sack rate allowed and quarterback hit rate allowed are combined into a standardized protection score, with lower rates representing stronger performance.

Offensive line continuity is left for the team strength stage because the historical continuity feature describes roster retention entering the completed season rather than the upcoming 2026 season.

In [53]:
historical_team_features = pl.read_parquet(
    PROCESSED_DIR / "historical_team_features.parquet"
)

In [54]:
ol_2026 = (
    historical_team_features
    .filter(
        pl.col("season") == 2025
    )
    .select([
        "team",
        "sack_rate_allowed",
        "qb_hit_rate_allowed"
    ])
)

In [55]:
ol_2026 = ol_2026.with_columns([
    (
        -0.60 * (
            (
                pl.col("sack_rate_allowed")
                - pl.col("sack_rate_allowed").mean()
            )
            / pl.col("sack_rate_allowed").std()
        )
        -0.40 * (
            (
                pl.col("qb_hit_rate_allowed")
                - pl.col("qb_hit_rate_allowed").mean()
            )
            / pl.col("qb_hit_rate_allowed").std()
        )
    ).alias("ol_protection_strength")
])

In [56]:
ol_2026_strength = (
    ol_2026
    .select([
        "team",
        "ol_protection_strength"
    ])
    .sort(
        "ol_protection_strength",
        descending=True
    )
)

The 2026 projection uses the most recent season's pass protection performance as the primary OL signal. Historical continuity is not carried forward directly because the 2025 continuity value describes roster retention entering 2025 rather than entering 2026.

Current offseason continuity can be incorporated later when 2026 roster movement is combined with the team strength model.

In [57]:
ol_2026_strength.write_parquet(
    PROCESSED_DIR / "2026_ol_strength.parquet"
)

## Defensive Player Projections

Defensive players are projected using recent individual production and playing time.

Rather than building separate models for every defensive position, players are grouped into the front seven and secondary. Recent sacks, quarterback hits, tackles, interceptions, passes defended, and defensive snaps are used to estimate each player's recent defensive contribution before aggregating players to the team level.

In [58]:
defensive_history = (
    projection_rows
    .filter(
        pl.col("projection_position").is_in([
            "DL",
            "LB",
            "CB",
            "S"
        ])
    )
)

In [59]:
defensive_projection_rows = defensive_history.with_columns([
    weighted_history_expr("def_tackles_solo")
    .alias("weighted_tackles"),

    weighted_history_expr("def_sacks")
    .alias("weighted_sacks"),

    weighted_history_expr("def_qb_hits")
    .alias("weighted_qb_hits"),

    weighted_history_expr("def_interceptions")
    .alias("weighted_interceptions"),

    weighted_history_expr("def_pass_defended")
    .alias("weighted_pass_defended"),

    weighted_history_expr("defense_snaps")
    .alias("weighted_defense_snaps")
])

In [60]:
defensive_2026_roster = (
    current_2026_rosters
    .filter(
        pl.col("position").is_in([
            "DL",
            "LB",
            "DB"
        ])
        & pl.col("gsis_id").is_not_null()
    )
    .sort([
        "gsis_id",
        "week"
    ])
    .group_by("gsis_id")
    .agg([
        pl.col("team").last().alias("team"),
        pl.col("full_name").last().alias(
            "player_display_name"
        ),
        pl.col("position").last().alias("position")
    ])
    .rename({
        "gsis_id": "player_id"
    })
    .with_columns(
        pl.when(
            pl.col("position").is_in([
                "DL",
                "LB"
            ])
        )
        .then(pl.lit("Front Seven"))
        .when(
            pl.col("position") == "DB"
        )
        .then(pl.lit("Secondary"))
        .otherwise(None)
        .alias("defensive_group")
    )
)

In [61]:
defensive_2026 = defensive_2026_roster

for lag, source_season in [
    (1, 2025),
    (2, 2024),
    (3, 2023)
]:
    lag_data = (
        player_history_source
        .filter(
            pl.col("season") == source_season
        )
        .select([
            "player_id",
            "def_tackles_solo",
            "def_sacks",
            "def_qb_hits",
            "def_interceptions",
            "def_pass_defended",
            "defense_snaps"
        ])
        .rename({
            "def_tackles_solo":
                f"lag{lag}_def_tackles_solo",
            "def_sacks":
                f"lag{lag}_def_sacks",
            "def_qb_hits":
                f"lag{lag}_def_qb_hits",
            "def_interceptions":
                f"lag{lag}_def_interceptions",
            "def_pass_defended":
                f"lag{lag}_def_pass_defended",
            "defense_snaps":
                f"lag{lag}_defense_snaps"
        })
    )

    defensive_2026 = defensive_2026.join(
        lag_data,
        on="player_id",
        how="left"
    )

In [62]:
defensive_2026 = defensive_2026.with_columns([
    weighted_history_expr(
        "def_tackles_solo"
    ).alias("weighted_tackles"),

    weighted_history_expr(
        "def_sacks"
    ).alias("weighted_sacks"),

    weighted_history_expr(
        "def_qb_hits"
    ).alias("weighted_qb_hits"),

    weighted_history_expr(
        "def_interceptions"
    ).alias("weighted_interceptions"),

    weighted_history_expr(
        "def_pass_defended"
    ).alias("weighted_pass_defended"),

    weighted_history_expr(
        "defense_snaps"
    ).alias("weighted_defense_snaps")
])

In [63]:
defensive_2026 = defensive_2026.with_columns(
    pl.when(
        pl.col("defensive_group") == "Front Seven"
    )
    .then(
        0.5
        * pl.col("weighted_tackles").fill_null(0)
        + 6.0
        * pl.col("weighted_sacks").fill_null(0)
        + 2.0
        * pl.col("weighted_qb_hits").fill_null(0)
    )
    .when(
        pl.col("defensive_group") == "Secondary"
    )
    .then(
        0.4
        * pl.col("weighted_tackles").fill_null(0)
        + 12.0
        * pl.col("weighted_interceptions").fill_null(0)
        + 3.0
        * pl.col("weighted_pass_defended").fill_null(0)
    )
    .otherwise(None)
    .alias("defensive_production_score")
)

In [64]:
defensive_2026 = defensive_2026.with_columns(
    pl.col("defensive_production_score")
    .rank(
        method="ordinal",
        descending=True
    )
    .over([
        "team",
        "defensive_group"
    ])
    .alias("group_rank")
)

In [65]:
team_defensive_strength = (
    defensive_2026
    .filter(
        (
            (pl.col("defensive_group") == "Front Seven")
            & (pl.col("group_rank") <= 7)
        )
        |
        (
            (pl.col("defensive_group") == "Secondary")
            & (pl.col("group_rank") <= 5)
        )
    )
    .group_by([
        "team",
        "defensive_group"
    ])
    .agg(
        pl.col("defensive_production_score")
        .sum()
        .alias("defensive_strength")
    )
    .pivot(
        values="defensive_strength",
        index="team",
        on="defensive_group"
    )
    .rename({
        "Front Seven": "front_seven_strength",
        "Secondary": "secondary_strength"
    })
)

In [66]:
team_defensive_strength = team_defensive_strength.with_columns([
    (
        (
            pl.col("front_seven_strength")
            - pl.col("front_seven_strength").mean()
        )
        / pl.col("front_seven_strength").std()
    ).alias("front_seven_strength_z"),

    (
        (
            pl.col("secondary_strength")
            - pl.col("secondary_strength").mean()
        )
        / pl.col("secondary_strength").std()
    ).alias("secondary_strength_z")
])

In [67]:
team_defensive_strength = (
    team_defensive_strength
    .select([
        "team",
        "front_seven_strength_z",
        "secondary_strength_z"
    ])
    .sort(
        "front_seven_strength_z",
        descending=True
    )
)

In [68]:
team_defensive_strength.write_parquet(
    PROCESSED_DIR / "2026_defensive_strength.parquet"
)

The final defensive ratings are standardized across the league so that 0 represents an average unit. Positive values indicate stronger projected returning production, while negative values indicate weaker projected production.

These ratings are intended to capture current defensive personnel strength rather than predict total defensive performance on their own. Team-level scheme, efficiency, continuity, and coaching context are incorporated later in the team strength model.

## 2026 Roster Continuity

Recent team performance is more informative when a meaningful portion of the roster remains intact.

Roster continuity measures the share of each team's current 2026 roster that was also on the same team in 2025. This provides a simple year-to-year roster change signal without attempting to assign subjective values to every offseason transaction.

In [69]:
rosters_2025 = (
    rosters
    .filter(
        (pl.col("season") == 2025)
        & pl.col("gsis_id").is_not_null()
    )
    .select([
        "team",
        "gsis_id"
    ])
    .unique()
)

rosters_2026 = (
    current_2026_rosters
    .filter(
        pl.col("gsis_id").is_not_null()
    )
    .sort([
        "gsis_id",
        "week"
    ])
    .group_by("gsis_id")
    .agg(
        pl.col("team").last().alias("team")
    )
)

In [70]:
returning_players_2026 = (
    rosters_2026
    .join(
        rosters_2025.with_columns(
            pl.lit(1).alias("returning_player")
        ),
        on=[
            "team",
            "gsis_id"
        ],
        how="left"
    )
    .with_columns(
        pl.col("returning_player")
        .fill_null(0)
    )
)

roster_continuity_2026 = (
    returning_players_2026
    .group_by("team")
    .agg([
        pl.len().alias("current_roster_players"),
        pl.col("returning_player")
        .sum()
        .alias("returning_players")
    ])
    .with_columns(
        (
            pl.col("returning_players")
            / pl.col("current_roster_players")
        ).alias("roster_continuity")
    )
    .select([
        "team",
        "roster_continuity"
    ])
)

In [71]:
roster_continuity_2026.write_parquet(
    PROCESSED_DIR / "2026_roster_continuity.parquet"
)

### Position-Group Continuity

Overall roster continuity treats every returning player equally. To better represent which parts of a team remain intact entering 2026, continuity is also measured separately for quarterback, offensive line, skill positions, front seven, and secondary.

A player counts as returning when they appear on the same team's 2025 and 2026 rosters. This provides a simple measure of positional stability without requiring subjective player value estimates.

In [72]:
position_groups = {
    "QB": ["QB"],
    "OL": ["OL"],
    "Skill": ["RB", "WR", "TE"],
    "Front Seven": ["DL", "LB"],
    "Secondary": ["DB"]
}

position_to_group = {
    position: group
    for group, positions in position_groups.items()
    for position in positions
}

roster_positions_2025 = (
    rosters
    .filter(
        (pl.col("season") == 2025)
        & pl.col("gsis_id").is_not_null()
        & pl.col("position").is_in(list(position_to_group))
    )
    .select([
        "team",
        "gsis_id",
        "position"
    ])
    .unique()
    .with_columns(
        pl.col("position")
        .replace(position_to_group)
        .alias("position_group")
    )
)

roster_positions_2026 = (
    current_2026_rosters
    .filter(
        pl.col("gsis_id").is_not_null()
        & pl.col("position").is_in(list(position_to_group))
    )
    .sort([
        "gsis_id",
        "week"
    ])
    .group_by("gsis_id")
    .agg([
        pl.col("team").last().alias("team"),
        pl.col("position").last().alias("position")
    ])
    .with_columns(
        pl.col("position")
        .replace(position_to_group)
        .alias("position_group")
    )
)

In [73]:
returning_by_group = (
    roster_positions_2026
    .join(
        roster_positions_2025
        .select([
            "team",
            "gsis_id"
        ])
        .with_columns(
            pl.lit(1).alias("returning_player")
        ),
        on=[
            "team",
            "gsis_id"
        ],
        how="left"
    )
    .with_columns(
        pl.col("returning_player").fill_null(0)
    )
)

position_group_continuity_2026 = (
    returning_by_group
    .group_by([
        "team",
        "position_group"
    ])
    .agg([
        pl.len().alias("current_players"),
        pl.col("returning_player")
        .sum()
        .alias("returning_players")
    ])
    .with_columns(
        (
            pl.col("returning_players")
            / pl.col("current_players")
        ).alias("continuity")
    )
    .select([
        "team",
        "position_group",
        "continuity"
    ])
    .pivot(
        values="continuity",
        index="team",
        on="position_group"
    )
    .rename({
        "QB": "qb_continuity",
        "OL": "ol_continuity",
        "Skill": "skill_continuity",
        "Front Seven": "front_seven_continuity",
        "Secondary": "secondary_continuity"
    })
)

## Final 2026 Player and Position-Group Projections

The final outputs summarize the player and position group information that will feed into the team strength model.

Quarterback is represented by projected EPA per dropback, skill positions by standardized recent production, offensive line by recent pass protection performance, and defense by standardized front seven and secondary production.

These ratings are not intended to predict team performance independently. Notebook 05 combines them with the historical team level features developed earlier in the project.

In [74]:
projection_outputs = {
    "QB": qb_2026_team_strength,
    "Skill Positions": team_skill_strength,
    "Offensive Line": ol_2026_strength,
    "Defense": team_defensive_strength
}

for name, df in projection_outputs.items():
    if isinstance(df, pl.DataFrame):
        rows = df.height
        teams = df["team"].n_unique()
    else:
        rows = len(df)
        teams = df["team"].nunique()

    print(
        f"{name}: "
        f"{rows} teams, "
        f"{teams} unique teams"
    )

QB: 32 teams, 32 unique teams
Skill Positions: 32 teams, 32 unique teams
Offensive Line: 32 teams, 32 unique teams
Defense: 32 teams, 32 unique teams
